In [12]:
# =============================================================================
# 1. IMPORT LIBRARY
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import xgboost as xgb

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 60)
print("PIPELINE ML - DETEKSI RISIKO KANKER")
print("=" * 60)


# =============================================================================
# 2. LOAD DATASET
# =============================================================================
print("\n[2] LOAD DATASET")
print("-" * 40)

URL = "https://raw.githubusercontent.com/Putraaa27/kanker/refs/heads/main/data.csv"
df = pd.read_csv(URL)

print(f"Shape dataset : {df.shape}")
print(f"\nInfo dataset:")
print(df.info())
print(f"\n5 Baris pertama:")
print(df.head())
print(f"\nNilai unik per kolom:")
print(df.nunique())


# =============================================================================
# 3. DATA CLEANING (VERSI PERBAIKAN)
# =============================================================================
print("\n[3] DATA CLEANING")
print("-" * 40)

# Hapus Kolom
df = df.dropna(axis=1, how='all')

# Hapus kolom ID
id_cols = [col for col in df.columns if col.lower() in ['id', 'patient_id', 'no', 'index', 'Unnamed: 32']]
df = df.drop(columns=[c for c in id_cols if c in df.columns])
print(f"Kolom tidak relevan dihapus. Shape sekarang: {df.shape}")

# Encoding kolom kategorikal
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"  - '{col}' di-encode.")

# 4. Tangani Missing Values
# Amankan nama kolom SEBELUM imputasi
current_columns = df.columns

imputer = SimpleImputer(strategy='median')
# Lakukan fit_transform
imputed_data = imputer.fit_transform(df)

# Buat DataFrame baru dengan jumlah kolom yang sesuai hasil imputer
df = pd.DataFrame(imputed_data, columns=current_columns)

print(f"Imputasi selesai. Shape akhir: {df.shape}")


# =============================================================================
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# =============================================================================
print("\n[4] EXPLORATORY DATA ANALYSIS")
print("-" * 40)

# Statistik deskriptif
print("Statistik deskriptif:")
print(df.describe())

# Distribusi target
print(f"\nDistribusi target '{target_col}':")
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True).round(3))

# --- Visualisasi ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Distribusi target
target_counts = df[target_col].value_counts()
axes[0].bar(target_counts.index.astype(str), target_counts.values, color=['#2196F3', '#F44336', '#4CAF50'])
axes[0].set_title(f'Distribusi Target: {target_col}', fontsize=13)
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Plot 2: Heatmap korelasi (ambil 15 fitur teratas agar tidak crowded)
feature_cols = [c for c in df.columns if c != target_col]
top_features = df[feature_cols].corrwith(df[target_col]).abs().nlargest(min(15, len(feature_cols))).index.tolist()
corr_matrix = df[top_features + [target_col]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[1], cmap='coolwarm',
            annot=len(top_features) <= 10, fmt='.2f', linewidths=0.5,
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Heatmap Korelasi (Top Fitur)', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

# Plot 3: Top 10 fitur berkorelasi dengan target
top10 = df[feature_cols].corrwith(df[target_col]).abs().nlargest(10)
top10.sort_values().plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 10 Fitur (Korelasi dengan Target)', fontsize=13)
axes[2].set_xlabel('Korelasi Absolut')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=100, bbox_inches='tight')
plt.close()
print("Visualisasi EDA disimpan: eda_analysis.png")

# Insight EDA
print("\n[Insight EDA]")
print(f"  - Dataset memiliki {df.shape[0]} sampel dan {df.shape[1]} fitur (termasuk target)")
print(f"  - Target '{target_col}' memiliki {df[target_col].nunique()} kelas unik")
imbalance = target_counts.max() / target_counts.min()
print(f"  - Rasio kelas max/min: {imbalance:.2f} ({'Imbalanced' if imbalance > 1.5 else 'Balanced'})")
print(f"  - Fitur paling berkorelasi dengan target: {top10.index[0]} ({top10.values[0]:.3f})")


# =============================================================================
# 5. FEATURE ENGINEERING
# =============================================================================
print("\n[5] FEATURE ENGINEERING")
print("-" * 40)

# Pisahkan fitur dan target
X = df.drop(columns=[target_col])
y = df[target_col]

# Jika target kontinu (regression problem), konversi ke klasifikasi
if y.nunique() > 10:
    print(f"Target memiliki {y.nunique()} nilai unik - mengkonversi ke klasifikasi dengan quantile binning")
    y = pd.qcut(y, q=3, labels=[0, 1, 2]).astype(int)
    print(f"Kelas baru setelah binning: {y.unique()}")
else:
    # Pastikan target adalah integer
    y = y.astype(int)
    # Re-encode target agar dimulai dari 0
    unique_vals = sorted(y.unique())
    val_map = {v: i for i, v in enumerate(unique_vals)}
    y = y.map(val_map)
    print(f"Target di-remap: {val_map}")

print(f"\nShape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"Distribusi y:\n{y.value_counts()}")

FEATURE_NAMES = X.columns.tolist()
N_FEATURES = len(FEATURE_NAMES)
N_CLASSES = y.nunique()
print(f"\nJumlah fitur  : {N_FEATURES}")
print(f"Jumlah kelas  : {N_CLASSES}")

# Split data 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling (StandardScaler) diterapkan.")


# =============================================================================
# 6-7. MODELING & TRAINING
# =============================================================================
print("\n[6-7] MODELING & TRAINING")
print("-" * 40)

# Tentukan apakah binary atau multiclass
is_binary = N_CLASSES == 2
print(f"Tipe klasifikasi: {'Binary' if is_binary else 'Multiclass'} ({N_CLASSES} kelas)")

# --- Model 1: Logistic Regression ---
print("\nTraining Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    C=1.0,
    solver='lbfgs',
    multi_class='auto'
)
lr_model.fit(X_train_scaled, y_train)
print("  Logistic Regression selesai ditraining.")

# --- Model 2: Random Forest ---
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("  Random Forest selesai ditraining.")

# --- Model 3: XGBoost ---
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss' if not is_binary else 'logloss',
    random_state=RANDOM_STATE,
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train)
print("  XGBoost selesai ditraining.")


# =============================================================================
# 8. EVALUASI MODEL
# =============================================================================
print("\n[8] EVALUASI MODEL")
print("-" * 40)

def evaluate_model(model, X_test, y_test, model_name):
    """Fungsi evaluasi lengkap untuk satu model."""
    y_pred = model.predict(X_test)

    # ROC-AUC
    try:
        if is_binary:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            y_prob = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    except Exception:
        auc = None

    metrics = {
        'Model'    : model_name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall'   : recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score' : f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'ROC-AUC'  : auc if auc is not None else np.nan,
    }
    return metrics, y_pred


models = {
    'Logistic Regression': (lr_model, X_test_scaled),
    'Random Forest'      : (rf_model, X_test_scaled),
    'XGBoost'            : (xgb_model, X_test_scaled),
}

results = []
predictions = {}

for name, (model, X_t) in models.items():
    metrics, y_pred = evaluate_model(model, X_t, y_test, name)
    results.append(metrics)
    predictions[name] = y_pred

    # Ambil nilai AUC agar kode lebih rapi
    auc_val = metrics['ROC-AUC']
    # Logika tampilan: jika bukan NaN, format ke 4 desimal, jika NaN tulis 'N/A'
    auc_display = f"{auc_val:.4f}" if not np.isnan(auc_val) else "N/A"

    print(f"\n[{name}]")
    print(f"  Accuracy  : {metrics['Accuracy']:.4f}")
    print(f"  Precision : {metrics['Precision']:.4f}")
    print(f"  Recall    : {metrics['Recall']:.4f}")
    print(f"  F1-Score  : {metrics['F1-Score']:.4f}")
    print(f"  ROC-AUC   : {auc_display}") # <-- Gunakan variabel bantuan di sini
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred, zero_division=0)}")

# Tabel perbandingan
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.round(4)
print("\n" + "=" * 60)
print("TABEL PERBANDINGAN MODEL")
print("=" * 60)
print(results_df.to_string())

# Visualisasi perbandingan & confusion matrix
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Baris 1: Confusion Matrix masing-masing model
for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=axes[0, idx], colorbar=False, cmap='Blues')
    axes[0, idx].set_title(f'Confusion Matrix\n{name}', fontsize=11)

# Baris 2: Perbandingan metrik
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF5722']
model_names = list(predictions.keys())

for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [results_df.loc[name, m] for m in metrics_to_plot]
    axes[1, 0].bar(x + i * width, vals, width, label=name, color=color, alpha=0.85)

axes[1, 0].set_xticks(x + width)
axes[1, 0].set_xticklabels(metrics_to_plot)
axes[1, 0].set_ylim(0, 1.1)
axes[1, 0].set_title('Perbandingan Metrik Model', fontsize=12)
axes[1, 0].legend()
axes[1, 0].set_ylabel('Score')

# ROC-AUC bar chart
auc_vals = results_df['ROC-AUC'].values
axes[1, 1].bar(model_names, auc_vals, color=colors, alpha=0.85)
axes[1, 1].set_title('ROC-AUC Score per Model', fontsize=12)
axes[1, 1].set_ylabel('AUC')
axes[1, 1].set_ylim(0, 1.1)
for i, v in enumerate(auc_vals):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Feature importance (dari Random Forest)
feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURE_NAMES).nlargest(10)
feat_imp.sort_values().plot(kind='barh', ax=axes[1, 2], color='steelblue')
axes[1, 2].set_title('Top 10 Feature Importance (Random Forest)', fontsize=12)
axes[1, 2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=100, bbox_inches='tight')
plt.close()
print("\nVisualisasi evaluasi disimpan: model_evaluation.png")


# =============================================================================
# 9. MODEL SELECTION
# =============================================================================
print("\n[9] MODEL SELECTION")
print("-" * 40)

# Kriteria pemilihan: F1-Score (weighted) sebagai metrik utama
# Alasan: F1-Score menyeimbangkan Precision dan Recall, cocok untuk data medis
# dimana false negative (melewatkan kasus kanker) sama berbahayanya dengan false positive
best_model_name = results_df['F1-Score'].idxmax()
best_score = results_df.loc[best_model_name, 'F1-Score']

print(f"Kriteria seleksi: F1-Score (Weighted)")
print(f"Model terbaik   : {best_model_name}")
print(f"F1-Score terbaik: {best_score:.4f}")
print(f"\nAlasan: F1-Score dipilih karena menyeimbangkan Precision & Recall.")
print(f"Dalam konteks medis, penting untuk meminimalkan false negative (kanker tidak terdeteksi).")

# Ambil model terbaik
model_map = {
    'Logistic Regression': lr_model,
    'Random Forest'      : rf_model,
    'XGBoost'            : xgb_model,
}
best_model = model_map[best_model_name]

# Simpan model terbaik (pickle) beserta scaler
with open('best_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'scaler': scaler, 'features': FEATURE_NAMES}, f)
print("\nModel terbaik disimpan: best_model.pkl")


# =============================================================================
# 10. EXPORT KE ONNX
# =============================================================================
print("\n[10] EXPORT KE ONNX")
print("-" * 40)

# Buat pipeline sklearn lengkap (scaler + model) untuk konversi ONNX
# Khusus XGBoost: gunakan pipeline sklearn wrapper
if best_model_name == 'XGBoost':
    # XGBoost perlu wrapper agar bisa dikonversi via skl2onnx
    from sklearn.pipeline import Pipeline as SKPipeline
    from xgboost import XGBClassifier

    best_pipeline = SKPipeline([
        ('scaler', scaler),
        ('model', xgb_model)
    ])
    # Re-fit pipeline pada data training
    best_pipeline.fit(X_train, y_train)
    model_to_convert = best_pipeline
else:
    # Untuk LR dan RF: buat pipeline
    from sklearn.pipeline import Pipeline as SKPipeline
    best_pipeline = SKPipeline([
        ('scaler', scaler),
        ('model', best_model)
    ])
    best_pipeline.fit(X_train, y_train)
    model_to_convert = best_pipeline

# Definisikan input type untuk ONNX
# Input: float32, shape [None, N_FEATURES] (None = batch size dinamis)
initial_type = [('float_input', FloatTensorType([None, N_FEATURES]))]

print(f"Mengonversi model '{best_model_name}' ke ONNX...")
print(f"Input shape: [batch_size, {N_FEATURES}]")

try:
    onnx_model = convert_sklearn(
        model_to_convert,
        initial_types=initial_type,
        target_opset=12,
        options={id(model_to_convert): {'zipmap': False}}
    )
    with open('model.onnx', 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print("Model ONNX berhasil disimpan: model.onnx")
    print(f"Ukuran file: {os.path.getsize('model.onnx') / 1024:.1f} KB")
except Exception as e:
    print(f"Gagal konversi pipeline: {e}")
    print("Mencoba konversi hanya model (tanpa pipeline)...")
    # Fallback: konversi hanya model tanpa scaler, lakukan scaling manual di inference
    try:
        if best_model_name == 'XGBoost':
            raise Exception("XGBoost memerlukan pendekatan lain")

        onnx_model = convert_sklearn(
            best_model,
            initial_types=initial_type,
            target_opset=12,
            options={id(best_model): {'zipmap': False}}
        )
        with open('model.onnx', 'wb') as f:
            f.write(onnx_model.SerializeToString())
        print("Model ONNX (tanpa scaler) disimpan: model.onnx")
        ONNX_WITH_SCALER = False
    except Exception as e2:
        print(f"Gagal konversi model saja: {e2}")
        # Fallback terakhir: gunakan Random Forest (paling kompatibel)
        print("Fallback: menggunakan Random Forest untuk ONNX export...")
        rf_pipeline = SKPipeline([('scaler', scaler), ('model', rf_model)])
        rf_pipeline.fit(X_train, y_train)
        onnx_model = convert_sklearn(
            rf_pipeline,
            initial_types=initial_type,
            target_opset=12,
            options={id(rf_pipeline): {'zipmap': False}}
        )
        with open('model.onnx', 'wb') as f:
            f.write(onnx_model.SerializeToString())
        print("Model ONNX (Random Forest) disimpan: model.onnx")
        best_model_name = 'Random Forest (Fallback ONNX)'


# =============================================================================
# 11. SIMULASI INFERENCE
# =============================================================================
print("\n[11] SIMULASI INFERENCE")
print("-" * 40)

# Buat dummy input dari data test (1 sampel)
sample_idx = 0
sample_raw = X_test.iloc[[sample_idx]].values  # shape (1, N_FEATURES)
sample_scaled = X_test_scaled[[sample_idx]]     # shape (1, N_FEATURES)
true_label = y_test.iloc[sample_idx]

print(f"Sampel input (raw, sebelum scaling):")
print(sample_raw)
print(f"Label asli: {true_label}")

# --- Prediksi menggunakan model sklearn asli ---
pred_sklearn = best_model.predict(sample_scaled)[0]
prob_sklearn = best_model.predict_proba(sample_scaled)[0]
print(f"\n[Sklearn Inference]")
print(f"  Prediksi kelas : {pred_sklearn}")
print(f"  Probabilitas   : {[round(p, 4) for p in prob_sklearn]}")

# --- Prediksi menggunakan ONNX Runtime ---
print(f"\n[ONNX Runtime Inference]")
try:
    sess = rt.InferenceSession('model.onnx')
    input_name = sess.get_inputs()[0].name
    output_label = sess.get_outputs()[0].name
    output_prob = sess.get_outputs()[1].name if len(sess.get_outputs()) > 1 else None

    # Input ONNX harus float32
    onnx_input = sample_raw.astype(np.float32)  # pipeline sudah include scaler

    onnx_outputs = sess.run(None, {input_name: onnx_input})
    pred_onnx = onnx_outputs[0][0]

    if output_prob and len(onnx_outputs) > 1:
        prob_onnx = onnx_outputs[1][0]
    else:
        prob_onnx = [None]

    print(f"  Input name     : {input_name}")
    print(f"  Prediksi kelas : {pred_onnx}")
    print(f"  Probabilitas   : {[round(float(p), 4) for p in prob_onnx] if prob_onnx[0] is not None else 'N/A'}")
    print(f"  Label asli     : {true_label}")
    print(f"  Konsistensi    : {'✓ KONSISTEN' if int(pred_sklearn) == int(pred_onnx) else '✗ BERBEDA'}")
except Exception as e:
    print(f"  Error ONNX inference: {e}")
    print("  Pastikan model.onnx berhasil dibuat.")


# =============================================================================
# 12. OUTPUT UNTUK WEB (Next.js + TypeScript)
# =============================================================================
print("\n[12] OUTPUT UNTUK WEB (Next.js + TypeScript)")
print("-" * 40)

# Simpan metadata untuk web integration
web_metadata = {
    "model_info": {
        "best_model"  : best_model_name,
        "n_features"  : N_FEATURES,
        "n_classes"   : N_CLASSES,
        "feature_names": FEATURE_NAMES,
        "onnx_file"   : "model.onnx",
        "input_dtype" : "float32",
        "input_shape" : [1, N_FEATURES]
    },
    "inference_guide": {
        "description"  : "Kirim array fitur numerik dalam format float32",
        "input_format" : f"Array 1D dengan {N_FEATURES} elemen numerik",
        "preprocessing": "Data SUDAH di-scale di dalam ONNX model (pipeline)",
        "output_format": {
            "label"      : "integer - kelas prediksi (0, 1, ...)",
            "probability": f"array float dengan {N_CLASSES} elemen"
        }
    },
    "example_input": {
        "json_payload": {
            "features": sample_raw[0].tolist()
        },
        "features_description": {
            name: f"Feature ke-{i+1}" for i, name in enumerate(FEATURE_NAMES)
        }
    },
    "example_output": {
        "predicted_class" : int(pred_sklearn),
        "probabilities"   : [round(float(p), 4) for p in prob_sklearn],
        "class_labels"    : list(range(N_CLASSES))
    }
}

with open('web_metadata.json', 'w') as f:
    json.dump(web_metadata, f, indent=2, default=str)

print("Metadata web disimpan: web_metadata.json")
print("\n--- PANDUAN INTEGRASI WEB (Next.js + TypeScript) ---\n")
print(f"1. ONNX Model: model.onnx")
print(f"   - Install: npm install onnxruntime-web")
print(f"   - Input  : Float32Array dengan {N_FEATURES} elemen")
print(f"   - Input sudah termasuk preprocessing (StandardScaler)")
print()
print(f"2. Contoh JSON Input dari Frontend:")
example_features = sample_raw[0].tolist()[:5]
print(json.dumps({
    "features": example_features + ['...', f'(total {N_FEATURES} nilai)']
}, indent=2))
print()
print(f"3. Contoh Output dari Model ONNX:")
print(json.dumps({
    "predicted_class" : int(pred_sklearn),
    "class_meaning"   : f"Kelas {int(pred_sklearn)} dari {N_CLASSES} kelas",
    "probabilities"   : {
        f"class_{i}": round(float(p), 4) for i, p in enumerate(prob_sklearn)
    },
    "high_risk"       : bool(int(pred_sklearn) > 0)
}, indent=2))
print()

print("""
4. Contoh Kode TypeScript (Next.js):

   import * as ort from 'onnxruntime-web';

   export async function predictCancerRisk(features: number[]) {
     const session = await ort.InferenceSession.create('/model.onnx');

     // Input harus Float32Array
     const inputTensor = new ort.Tensor(
       'float32',
       new Float32Array(features),
       [1, features.length]  // [batch_size, n_features]
     );

     const feeds: Record<string, ort.Tensor> = {
       float_input: inputTensor  // nama sesuai ONNX input name
     };

     const results = await session.run(feeds);

     const label = results['label'].data[0];        // kelas prediksi
     const probs = results['probabilities'].data;   // probabilitas tiap kelas

     return {
       predictedClass: Number(label),
       probabilities: Array.from(probs as Float32Array),
       isHighRisk: Number(label) > 0
     };
   }
""")


# =============================================================================
# RINGKASAN AKHIR
# =============================================================================
print("\n" + "=" * 60)
print("RINGKASAN PIPELINE SELESAI")
print("=" * 60)
print(f"Dataset        : {df.shape[0]} sampel, {N_FEATURES} fitur")
print(f"Jumlah kelas   : {N_CLASSES}")
print(f"Model terbaik  : {best_model_name}")
print(f"F1-Score       : {results_df['F1-Score'].max():.4f}")
print(f"\nFile yang dihasilkan:")
print(f"  - model.onnx          : Model untuk produksi (web/Next.js)")
print(f"  - best_model.pkl      : Model sklearn + scaler (backup)")
print(f"  - web_metadata.json   : Metadata untuk integrasi frontend")
print(f"  - eda_analysis.png    : Visualisasi EDA")
print(f"  - model_evaluation.png: Visualisasi evaluasi model")
print("=" * 60)

PIPELINE ML - DETEKSI RISIKO KANKER

[2] LOAD DATASET
----------------------------------------
Shape dataset : (569, 33)

Info dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12